# 00 Prepare Antibiotic Inputs

Build canonical antibiotic CSV inputs from `antibiotic_list3.xlsx`.

Outputs:

- `antibiotic_synonym_lookup.csv`: two-column synonym lookup mirroring `bacteria_synonym_lookup.csv`.
- `antibiotic_coarsening_lookup.csv`: one row per canonical antibiotic with class and category metadata.


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display


## Configuration


In [ ]:
SOURCE_XLSX = Path("antibiotic_list3.xlsx")
SOURCE_SHEET = "Sheet1"

ANTIBIOTIC_SYNONYM_OUTPUT = Path("antibiotic_synonym_lookup.csv")
ANTIBIOTIC_COARSENING_OUTPUT = Path("antibiotic_coarsening_lookup.csv")

CANONICAL_COL = "Antibiotic"
SYNONYM_COLS = ["alternative spelling", "Unnamed: 2", "Unnamed: 3", "Unnamed: 4"]
COARSENING_COLS = [
    "Antibiotic",
    "Class",
    "Simplified categories",
    "High level categories",
    "Origin",
]


## Load and Validate Source Workbook


In [ ]:
source_df = pd.read_excel(SOURCE_XLSX, sheet_name=SOURCE_SHEET)
source_df.columns = [str(col).strip() for col in source_df.columns]

required_cols = {CANONICAL_COL, *SYNONYM_COLS, *COARSENING_COLS}
missing_cols = required_cols - set(source_df.columns)
if missing_cols:
    raise KeyError(f"{SOURCE_XLSX} is missing required columns: {sorted(missing_cols)}")

source_df = source_df.dropna(subset=[CANONICAL_COL]).copy()
source_df[CANONICAL_COL] = source_df[CANONICAL_COL].astype(str).str.strip()
source_df = source_df.loc[source_df[CANONICAL_COL] != ""].copy()

print(f"Loaded {len(source_df):,} antibiotic rows from {SOURCE_XLSX}")
display(source_df.head(10))


## Build Antibiotic Synonym Lookup

The output mirrors the bacteria synonym lookup shape: one synonym row per canonical antibiotic. The canonical antibiotic name is included as its own synonym so downstream query expansion can use a single lookup table.


In [ ]:
def clean_text(value):
    if pd.isna(value):
        return None
    text = " ".join(str(value).strip().split())
    return text or None


synonym_rows = []
for _, row in source_df.iterrows():
    canonical = clean_text(row[CANONICAL_COL])
    if canonical is None:
        continue
    variants = [canonical]
    variants.extend(clean_text(row[col]) for col in SYNONYM_COLS)
    for variant in variants:
        if variant is not None:
            synonym_rows.append(
                {
                    "antibiotic synonym": variant,
                    "antibiotic name": canonical,
                }
            )

antibiotic_synonym_df = (
    pd.DataFrame(synonym_rows)
    .drop_duplicates()
    .sort_values(["antibiotic name", "antibiotic synonym"])
    .reset_index(drop=True)
)

antibiotic_synonym_df.to_csv(ANTIBIOTIC_SYNONYM_OUTPUT, index=False)
print(f"Saved {len(antibiotic_synonym_df):,} synonym rows to {ANTIBIOTIC_SYNONYM_OUTPUT}")
display(antibiotic_synonym_df.head(20))


## Build Antibiotic Coarsening Lookup

This table stores the canonical antibiotic-to-category mapping used for grouped analyses and coarser visualizations.


In [ ]:
antibiotic_coarsening_df = source_df[COARSENING_COLS].copy()
for col in COARSENING_COLS:
    antibiotic_coarsening_df[col] = antibiotic_coarsening_df[col].map(clean_text)

antibiotic_coarsening_df = (
    antibiotic_coarsening_df
    .dropna(subset=["Antibiotic"])
    .drop_duplicates(subset=["Antibiotic"], keep="first")
    .sort_values("Antibiotic")
    .reset_index(drop=True)
)

antibiotic_coarsening_df.to_csv(ANTIBIOTIC_COARSENING_OUTPUT, index=False)
print(f"Saved {len(antibiotic_coarsening_df):,} coarsening rows to {ANTIBIOTIC_COARSENING_OUTPUT}")
display(antibiotic_coarsening_df.head(20))


## Quick Checks


In [ ]:
print("Canonical antibiotics:", antibiotic_coarsening_df["Antibiotic"].nunique())
print("Synonym variants:", antibiotic_synonym_df["antibiotic synonym"].nunique())
print("Classes:", antibiotic_coarsening_df["Class"].nunique(dropna=True))
print("Simplified categories:", antibiotic_coarsening_df["Simplified categories"].nunique(dropna=True))
print("High level categories:", antibiotic_coarsening_df["High level categories"].nunique(dropna=True))

duplicate_synonyms = antibiotic_synonym_df.groupby("antibiotic synonym")["antibiotic name"].nunique()
duplicate_synonyms = duplicate_synonyms[duplicate_synonyms > 1]
if duplicate_synonyms.empty:
    print("No synonym variants map to multiple canonical antibiotics.")
else:
    print("Synonym variants mapping to multiple canonical antibiotics:")
    display(duplicate_synonyms.reset_index(name="canonical_count"))
